In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import os
from datetime import datetime, timedelta
import lightgbm as lgb
import joblib
import matplotlib.cm as cm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta
import lightgbm as lgb
import joblib
import matplotlib.cm as cm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

In [2]:
# =============================
# MultiColumnLabelEncoder
# =============================
class MultiColumnLabelEncoder:
    def __init__(self, categorical_cols, unknown_token="unknown"):
        self.categorical_cols = categorical_cols
        self.unknown_token = unknown_token
        self.encoders = {}

    def fit(self, df: pd.DataFrame):
        for col in self.categorical_cols:
            le = LabelEncoder()
            values = df[col].fillna(self.unknown_token).astype(str)
            classes = values.unique().tolist()
            if self.unknown_token not in classes:
                classes.append(self.unknown_token)
            le.fit(classes)
            self.encoders[col] = le
        return self

    def transform(self, df: pd.DataFrame):
        df = df.copy()
        for col, le in self.encoders.items():
            values = df[col].fillna(self.unknown_token).astype(str)
            # Những giá trị không nằm trong classes_ được map sang unknown_token
            values = values.where(values.isin(le.classes_), self.unknown_token)
            df[col] = le.transform(values)
        return df

    def fit_transform(self, df: pd.DataFrame):
        self.fit(df)
        return self.transform(df)

    def save(self, path: str):
        joblib.dump({
            "categorical_cols": self.categorical_cols,
            "unknown_token": self.unknown_token,
            "encoders": self.encoders
        }, path)

    @classmethod
    def load(cls, path: str):
        data = joblib.load(path)
        obj = cls(
            categorical_cols=data["categorical_cols"],
            unknown_token=data["unknown_token"]
        )
        obj.encoders = data["encoders"]
        return obj

# MERGE DATA TRAINING

In [3]:
model_dir = "models_1578_csv"
os.makedirs(model_dir, exist_ok=True)

In [4]:
def load_data(path):
    data_df = pd.read_csv(path)
    return data_df

In [5]:
dataset_all_building = "data_1578_csv"
list_building = [
    f for f in os.listdir(dataset_all_building)
]
print(list_building)

['Eagle_assembly_Candice', 'Eagle_assembly_Estelle', 'Eagle_assembly_Ian', 'Eagle_assembly_Josie', 'Eagle_education_Luther', 'Eagle_education_Samantha', 'Eagle_education_Shanna', 'Eagle_education_Wesley', 'Eagle_education_Will', 'Eagle_lodging_Andy', 'Eagle_lodging_Stephanie', 'Eagle_office_Donovan', 'Eagle_office_Efrain', 'Eagle_office_Elvis', 'Eagle_office_Francis', 'Eagle_office_Henriette', 'Eagle_office_Isidro', 'Eagle_office_Jeff', 'Eagle_office_Lane', 'Eagle_office_Mable', 'Eagle_office_Randolph', 'Eagle_office_Ryan', 'Eagle_office_Sonya', 'Eagle_office_Yadira', 'Eagle_public_Henry', 'Eagle_public_Minnie', 'Eagle_public_Ola', 'Eagle_public_Preston', 'Fox_assembly_Adrianne', 'Fox_assembly_Audrey', 'Fox_assembly_Cathy', 'Fox_assembly_Emma', 'Fox_assembly_Johnnie', 'Fox_assembly_Renna', 'Fox_education_Andre', 'Fox_education_Ashli', 'Fox_education_Claire', 'Fox_education_Claude', 'Fox_education_Delma', 'Fox_education_Dewayne', 'Fox_education_Dominique', 'Fox_education_Eldon', 'Fox_ed

In [6]:

# =============================
# CONFIG
# =============================
CAT_COLS = ["primaryspaceusage", "site_id", "building_id"]  # sửa typo

# =============================
# LOAD & CONCAT TRAIN DATA
# =============================
df_train_all = pd.DataFrame()
for building in list_building:
    path = f"{dataset_all_building}/{building}/train.csv"
    if os.path.exists(path):
        df = load_data(path)  # bạn định nghĩa hàm load_data()
        print("*"*10)
        print(f"Process building: {building}- {len(df)}", )
        df_train_all = pd.concat([df_train_all, df], ignore_index=True)
        print("Total: ", len(df_train_all))
    else:
        print(f"Train file not found for {building}")

df_train_all.to_csv(f"{dataset_all_building}/train.csv", index=False)

# =============================
# FIT ENCODER TRÊN TRAIN
# =============================
encoder = MultiColumnLabelEncoder(CAT_COLS)
df_train_encoded = encoder.fit_transform(df_train_all)
df_train_encoded.to_csv(f"{dataset_all_building}/train_encode.csv", index=False)
encoder.save(f"{model_dir}/categorical_encoder.pkl")  # save encoder sau khi fit

# =============================
# LOAD & CONCAT TEST DATA
# =============================
df_test_all = pd.DataFrame()
for building in list_building:
    path = f"{dataset_all_building}/{building}/test.csv"
    if os.path.exists(path):
        df = load_data(path)
        df_test_all = pd.concat([df_test_all, df], ignore_index=True)
        print("Total test: ", len(df_test_all))
    else:
        print(f"Test file not found for {building}")

df_test_all.to_csv(f"{dataset_all_building}/test.csv", index=False)

# =============================
# TRANSFORM TEST DATA
# =============================
encoder = MultiColumnLabelEncoder.load(f"{model_dir}/categorical_encoder.pkl")  # load encoder train
df_test_encoded = encoder.transform(df_test_all)  # chỉ transform
df_test_encoded.to_csv(f"{dataset_all_building}/test_encode.csv", index=False)

print("Train/Test encoding done ✅")

**********
Process building: Eagle_assembly_Candice- 13794
Total:  13794
**********
Process building: Eagle_assembly_Estelle- 13745
Total:  27539
**********
Process building: Eagle_assembly_Ian- 13916
Total:  41455
**********
Process building: Eagle_assembly_Josie- 13926
Total:  55381
**********
Process building: Eagle_education_Luther- 14406
Total:  69787
**********
Process building: Eagle_education_Samantha- 14471
Total:  84258
**********
Process building: Eagle_education_Shanna- 14072
Total:  98330
**********
Process building: Eagle_education_Wesley- 14041
Total:  112371
**********
Process building: Eagle_education_Will- 14314
Total:  126685
**********
Process building: Eagle_lodging_Andy- 13845
Total:  140530
**********
Process building: Eagle_lodging_Stephanie- 14020
Total:  154550
**********
Process building: Eagle_office_Donovan- 13866
Total:  168416
**********
Process building: Eagle_office_Efrain- 13814
Total:  182230
**********
Process building: Eagle_office_Elvis- 13869
Tota